In [4]:
import numpy as np
import pandas as pd
import torch 
from torch import nn
from torchvision.transforms import v2
from torchvision import datasets, transforms
from torch.utils.data import DataLoader as dl
import torch.nn.functional as F


# I will more or less be following PyTorch's intro documentation
# here as it is a good intro to the format. I will be adding other projects 
# that are more unique as I gain familiarity with Torch. 

In [5]:
train = datasets.FashionMNIST(root = '/Users/austinn/Desktop/learningml/learningpytorch/datapytorch',
                              train = True, 
                              download = True, 
                              transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True)]),
                              target_transform = v2.Lambda(lambda y: F.one_hot(torch.tensor(y), num_classes = 10).float())
                              )

test = datasets.FashionMNIST(root = '/Users/austinn/Desktop/learningml/learningpytorch/datapytorch',
                              train = False, 
                              download=True, 
                              transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True)])
                              )

train_loader = dl(train, batch_size = 64, shuffle = True)
test_loader = dl(test, batch_size = 64, shuffle = True)

PermissionError: [WinError 5] Access is denied: '/Users/austinn'

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
#On our mac, we have an MPS accelerator. Neat!
print(device)

cpu


In [21]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__() #inherit from nn.Module
        self.flatten = nn.Flatten() 
        #data flattening method for input into nn

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 233),  
            nn.ReLU(),
            nn.Linear(233,377),
            nn.ReLU(),
            nn.Linear(377,10)
        )
    #Trying fibonacci numbers as layer neuron counts


    def forward(self, x):
        x = self.flatten(x) 
        logits = self.linear_relu_stack(x) #Sequential passes elements in order. 
        return logits       #ready for softmax.


In [22]:
#instance of NeuralNetwork, move it to mps
func_model = NeuralNetwork().to(device)

In [ ]:
#testing network functionality


# randoms = torch.rand(1,28,28,device = device)   #dimension 0 is batch size. very convenient.
# out = func_model(randoms)
# random_probs = nn.Softmax(dim=1)(out)
# random_prediction = random_probs.argmax()
# print(random_prediction)

tensor(7, device='mps:0')


In [ ]:
alpha = .01
batch_size = 64
epochs = 10
optim = torch.optim.Adam(func_model.parameters(), lr=alpha)
lossce = nn.CrossEntropyLoss()

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (x,y) in enumerate(dataloader):
        #forward pass
        prediction = model(x)
        lossloc = loss_fn(prediction, y)

        #backpropagation
        lossloc.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch%100 == 0:
            print(f'{lossloc.item():.7f}, passes = [{batch*batch_size + len(x)}/{size}]')

    print(f'end loss { lossloc.item():.7f}')

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0,0

    with torch.no_grad():
        for x,y in dataloader:
            prediction = model(x)
            test_loss += loss_fn(prediction,y).item()
            correct = (prediction.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(f'Accuracy: {(100*correct):.2f}%, loss = {test_loss:.3f}')





In [ ]:
#running:
for epoch in range(epochs)
    print(15*"=")
    print(f'Epoch {epoch}')
    print(15*'=')
    train_loop(train_loader, func_model, lossce, optim)
    test_loop(test_loader, func_model, lossce)

print('now for my own versions!')
